In [36]:
import requests
import csv
import time
import pandas as pd

# Constants
BASE_URL = "https://api.foursquare.com/v3/places/"
OUTPUT_FILE = "restaurants_per_district.csv"
API_KEY = "fsq3791aJN4+bpzTQlUFQwCW0IL44wIzjt6/JnPIhhCNwdY="
HEADERS = {
    "Accept-Encoding": "gzip, deflate",
    "Accept": "*/*",
    "Connection": "keep-alive",
    "Authorization": API_KEY
}

# Load districts from CSV
districts_df = pd.read_excel("Districts.xlsx")  # Replace with your actual filename

# Function to fetch POIs for a given district using latitude & longitude
def fetch_pois(district, lat, lng, radius=3000, max_results=200):
    url = f"{BASE_URL}search"
    params = {
        "ll": f"{lat},{lng}",
        "radius": radius,
        "limit": 50,  # Get 50 results per request
        "fields": "fsq_id,name,geocodes,location,categories,rating,stats,popularity",
        "categories": "13065"  # Only fetch Restaurants
    }
    
    all_pois = []
    cursor = None  # Used for pagination

    while len(all_pois) < max_results:
        if cursor:
            params["cursor"] = cursor  # Add cursor to fetch the next page

        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching POIs for {district}: {response.status_code}")
            break  # Stop if an error occurs

        data = response.json()
        pois = data.get("results", [])
        all_pois.extend(pois)

        # Get the cursor for the next page
        cursor = data.get("context", {}).get("next")
        if not cursor:
            break  # Stop if there are no more pages

        time.sleep(1)  # Avoid hitting rate limits

    return all_pois[:max_results]  # Return only the requested number of results


# Save POIs to a CSV file
def save_to_csv(data):
    header = ['District', 'City', 'Latitude', 'Longtitude', 'POI Name', 'POI Latitude', 'POI Longitude', 'Category', 'Rating', 'Popularity']
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data)

# Main function
def main():
    all_pois = []
    for index, row in districts_df.iterrows():
        district = row['District_en']
        city = row['City']
        lat = row['Latitude']
        lng = row['Longtitude']
        
        print(f"Fetching POIs for {district}, {city}...")
        pois = fetch_pois(district, lat, lng,max_results=200)
        
        for poi in pois:
           categories = poi.get("categories", [])
           category_name = categories[0].get("name", "N/A") if categories else "N/A"

           all_pois.append([
           district, city, lat, lng,
           poi.get("name", "N/A"),
           poi.get("geocodes", {}).get("main", {}).get("latitude", "N/A"),
           poi.get("geocodes", {}).get("main", {}).get("longitude", "N/A"),
           category_name,  # Use the fixed category name
           poi.get("rating", "No rating"),
           poi.get("popularity", "No popularity data")
    ])

        
        time.sleep(1)  # Pause to avoid rate limits

    save_to_csv(all_pois)
    print(f"POIs saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


Fetching POIs for Al Khalidiyah Ash Shamaliyah Dist., الدمام...
Fetching POIs for 1st Industrial Dist., الدمام...
Fetching POIs for Al Fanar Dist., الدمام...
Fetching POIs for Al Athir Dist., الدمام...
Fetching POIs for Al Jalawiyah Dist., الدمام...
Fetching POIs for Al Nakhil Dist., الدمام...
Fetching POIs for Al Qazaz Dist., الدمام...
Fetching POIs for Al Badiyah Dist., الدمام...
Fetching POIs for Ad Dawasir Dist., الدمام...
Fetching POIs for Ad Dabab Dist., الدمام...
Fetching POIs for Al Anud Dist., الدمام...
Fetching POIs for Ash Shati Al Gharbi Dist., الدمام...
Fetching POIs for Al Manar Dist., الدمام...
Fetching POIs for An Nur Dist., الدمام...
Fetching POIs for As Salam Dist., الدمام...
Fetching POIs for Az Zuhur Dist., الدمام...
Fetching POIs for Al Muraikabat Dist., الدمام...
Fetching POIs for An Nahdah Dist., الدمام...
Fetching POIs for Dahiyat Al Malik Fahd Dist., الدمام...
Fetching POIs for As Saif Dist., الدمام...
Fetching POIs for As Sinaiyah Dist., الدمام...
Fetching POI

#### Using Closed bucket & closed date for Restaurants

In [11]:
import requests
import csv
import time
import pandas as pd

# Constants
BASE_URL = "https://api.foursquare.com/v3/places/"
OUTPUT_FILE = "new_restaurants_per_district.csv"
API_KEY = "fsq39JcmmM0pK04Rrdg11zFzV5i/HiUGTT2oOVPAphoIzSs="
HEADERS = {
    "Accept-Encoding": "gzip, deflate",
    "Accept": "*/*",
    "Connection": "keep-alive",
    "Authorization": API_KEY
}

# Load districts from CSV
districts_df = pd.read_excel("../Cleaned Data/Clean_Districts.xlsx")  

# Function to fetch POIs for a given district using latitude & longitude
def fetch_pois(district, lat, lng, radius=3000, max_results=200):
    url = f"{BASE_URL}search"
    params = {
        "ll": f"{lat},{lng}",
        "radius": radius,
        "limit": 50,  # Get 50 results per request
        "fields": "fsq_id,name,geocodes,location,categories,rating,stats,popularity,closed_bucket,date_closed",
        "categories": "13065"  # Only fetch Restaurants
    }
    
    all_pois = []
    cursor = None  # Used for pagination

    while len(all_pois) < max_results:
        if cursor:
            params["cursor"] = cursor  # Add cursor to fetch the next page

        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching POIs for {district}: {response.status_code}")
            break  # Stop if an error occurs

        data = response.json()
        pois = data.get("results", [])
        all_pois.extend(pois)

        # Get the cursor for the next page
        cursor = data.get("context", {}).get("next")
        if not cursor:
            break  # Stop if there are no more pages

        time.sleep(1)  # Avoid hitting rate limits

    return all_pois[:max_results]  # Return only the requested number of results


# Save POIs to a CSV file
def save_to_csv(data):
    header = [
        'District', 'City', 'Latitude', 'Longitude', 'POI Name', 'POI Latitude', 'POI Longitude', 
        'Category', 'Rating', 'Popularity', 'FSQ ID', 'Date_closed', 'Closed_bucket'
    ]
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data)

# Main function
def main():
    all_pois = []
    for index, row in districts_df.iterrows():
        district = row['District_en']
        city = row['City']
        lat = row['Latitude']
        lng = row['Longitude']  
        
        print(f"Fetching POIs for {district}, {city}...")
        pois = fetch_pois(district, lat, lng, max_results=200)
        
        for poi in pois:
            categories = poi.get("categories", [])
            category_name = categories[0].get("name", "N/A") if categories else "N/A"
            fsq_id = poi.get("fsq_id", "N/A")  # Get the FSQ ID
            closed_bucket = poi.get("closed_bucket", "N/A")  
            closed_date = poi.get("date_closed", "N/A") 

            all_pois.append([
                district, city, lat, lng,
                poi.get("name", "N/A"),
                poi.get("geocodes", {}).get("main", {}).get("latitude", "N/A"),
                poi.get("geocodes", {}).get("main", {}).get("longitude", "N/A"),
                category_name,
                poi.get("rating", "No rating"),
                poi.get("popularity", "No popularity data"),
                fsq_id,
                closed_bucket,
                closed_date,
              
            ])
        
        time.sleep(1)  # Pause to avoid rate limits

    save_to_csv(all_pois)
    print(f"POIs saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


Fetching POIs for Al Khalidiyah Ash Shamaliyah Dist., الدمام...
Fetching POIs for 1st Industrial Dist., الدمام...
Fetching POIs for Al Fanar Dist., الدمام...
Fetching POIs for Al Athir Dist., الدمام...
Fetching POIs for Al Jalawiyah Dist., الدمام...
Fetching POIs for Al Nakhil Dist., الدمام...
Fetching POIs for Al Qazaz Dist., الدمام...
Fetching POIs for Al Badiyah Dist., الدمام...
Fetching POIs for Ad Dawasir Dist., الدمام...
Fetching POIs for Ad Dabab Dist., الدمام...
Fetching POIs for Al Anud Dist., الدمام...
Fetching POIs for Ash Shati Al Gharbi Dist., الدمام...
Fetching POIs for Al Manar Dist., الدمام...
Fetching POIs for An Nur Dist., الدمام...
Fetching POIs for As Salam Dist., الدمام...
Fetching POIs for Az Zuhur Dist., الدمام...
Fetching POIs for Al Muraikabat Dist., الدمام...
Fetching POIs for An Nahdah Dist., الدمام...
Fetching POIs for Dahiyat Al Malik Fahd Dist., الدمام...
Fetching POIs for As Saif Dist., الدمام...
Fetching POIs for As Sinaiyah Dist., الدمام...
Fetching POI

#### Using Closed bucket & closed date for Coffee shops

In [1]:
import requests
import csv
import time
import pandas as pd

# Constants
BASE_URL = "https://api.foursquare.com/v3/places/"
OUTPUT_FILE = "new_coffeeShpos_per_district.csv"
API_KEY = "fsq39JcmmM0pK04Rrdg11zFzV5i/HiUGTT2oOVPAphoIzSs="
HEADERS = {
    "Accept-Encoding": "gzip, deflate",
    "Accept": "*/*",
    "Connection": "keep-alive",
    "Authorization": API_KEY
}

# Load districts from CSV
districts_df = pd.read_excel("../Cleaned Data/Clean_Districts.xlsx")  

# Function to fetch POIs for a given district using latitude & longitude
def fetch_pois(district, lat, lng, radius=3000, max_results=200):
    url = f"{BASE_URL}search"
    params = {
        "ll": f"{lat},{lng}",
        "radius": radius,
        "limit": 50,  # Get 50 results per request
        "fields": "fsq_id,name,geocodes,location,categories,rating,stats,popularity,closed_bucket,date_closed",
        "categories": "13032"  # For coffee shops
    }
    
    all_pois = []
    cursor = None  # Used for pagination

    while len(all_pois) < max_results:
        if cursor:
            params["cursor"] = cursor  # Add cursor to fetch the next page

        response = requests.get(url, headers=HEADERS, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching POIs for {district}: {response.status_code}")
            break  # Stop if an error occurs

        data = response.json()
        pois = data.get("results", [])
        all_pois.extend(pois)

        # Get the cursor for the next page
        cursor = data.get("context", {}).get("next")
        if not cursor:
            break  # Stop if there are no more pages

        time.sleep(1)  # Avoid hitting rate limits

    return all_pois[:max_results]  # Return only the requested number of results


# Save POIs to a CSV file
def save_to_csv(data):
    header = [
        'District', 'City', 'Latitude', 'Longitude', 'POI Name', 'POI Latitude', 'POI Longitude', 
        'Category', 'Rating', 'Popularity', 'FSQ ID', 'Date_closed', 'Closed_bucket'
    ]
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data)

# Main function
def main():
    all_pois = []
    for index, row in districts_df.iterrows():
        district = row['District_en']
        city = row['City']
        lat = row['Latitude']
        lng = row['Longitude']  
        
        print(f"Fetching POIs for {district}, {city}...")
        pois = fetch_pois(district, lat, lng, max_results=200)
        
        for poi in pois:
            categories = poi.get("categories", [])
            category_name = categories[0].get("name", "N/A") if categories else "N/A"
            fsq_id = poi.get("fsq_id", "N/A")  # Get the FSQ ID
            closed_bucket = poi.get("closed_bucket", "N/A")  
            closed_date = poi.get("date_closed", "N/A") 

            all_pois.append([
                district, city, lat, lng,
                poi.get("name", "N/A"),
                poi.get("geocodes", {}).get("main", {}).get("latitude", "N/A"),
                poi.get("geocodes", {}).get("main", {}).get("longitude", "N/A"),
                category_name,
                poi.get("rating", "No rating"),
                poi.get("popularity", "No popularity data"),
                fsq_id,
                closed_bucket,
                closed_date,
              
            ])
        
        time.sleep(1)  # Pause to avoid rate limits

    save_to_csv(all_pois)
    print(f"POIs saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


Fetching POIs for Al Khalidiyah Ash Shamaliyah Dist., الدمام...
Fetching POIs for 1st Industrial Dist., الدمام...
Fetching POIs for Al Fanar Dist., الدمام...
Fetching POIs for Al Athir Dist., الدمام...
Fetching POIs for Al Jalawiyah Dist., الدمام...
Fetching POIs for Al Nakhil Dist., الدمام...
Fetching POIs for Al Qazaz Dist., الدمام...
Fetching POIs for Al Badiyah Dist., الدمام...
Fetching POIs for Ad Dawasir Dist., الدمام...
Fetching POIs for Ad Dabab Dist., الدمام...
Fetching POIs for Al Anud Dist., الدمام...
Fetching POIs for Ash Shati Al Gharbi Dist., الدمام...
Fetching POIs for Al Manar Dist., الدمام...
Fetching POIs for An Nur Dist., الدمام...
Fetching POIs for As Salam Dist., الدمام...
Fetching POIs for Az Zuhur Dist., الدمام...
Fetching POIs for Al Muraikabat Dist., الدمام...
Fetching POIs for An Nahdah Dist., الدمام...
Fetching POIs for Dahiyat Al Malik Fahd Dist., الدمام...
Fetching POIs for As Saif Dist., الدمام...
Fetching POIs for As Sinaiyah Dist., الدمام...
Fetching POI